# Safebooru 메타데이터 크롤링
Safebooru API에서 메타데이터(태그 + URL)만 수집해 Parquet으로 저장합니다.
- **이미지는 저장하지 않음** — 학습 시 배치 단위로 임시 다운로드
- 저장 컬럼: `id`, `tags`, `file_url`, `sample_url`, `width`, `height`
- 중단 후 이어서 크롤링 가능

In [ ]:
import os
import time
import queue
import threading
import pandas as pd
from tqdm import tqdm
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException
from selenium.webdriver.chrome.options import Options
from fake_useragent import UserAgent

# --- 사용자 설정 ---
START_ID = 6654626
END_ID = 1
NUM_THREADS = 5
SAVE_INTERVAL = 1000
# 경로를 현재 사용자의 환경에 맞춰 수정했습니다.
SAVE_PATH = r"../data/metadata_html.parquet"

# --- 전역 변수 및 락 ---
buffer = []
save_lock = threading.Lock()
task_queue = queue.Queue(maxsize=1000)
ua = UserAgent()

def create_driver():
    chrome_options = Options()
    chrome_options.add_argument('--headless=new') 
    chrome_options.add_argument('--disable-gpu')
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    
    # --- Header 랜덤화 적용 ---
    chrome_options.add_argument(f'user-agent={ua.random}')
    
    # 자동화 탐지 방지 설정
    chrome_options.add_argument('--disable-blink-features=AutomationControlled')
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
    chrome_options.add_experimental_option('useAutomationExtension', False)
    
    prefs = {"profile.managed_default_content_settings.images": 2}
    chrome_options.add_experimental_option("prefs", prefs)
    
    driver = webdriver.Chrome(options=chrome_options)
    
    # 웹드라이버 인식 방지 자바스크립트 주입
    driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
        "source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
    })
    
    return driver

def save_buffer(pbar=None):
    global buffer
    if not buffer:
        return
    
    new_df = pd.DataFrame(buffer)
    os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)
    
    if os.path.exists(SAVE_PATH):
        try:
            existing_df = pd.read_parquet(SAVE_PATH)
            combined_df = pd.concat([existing_df, new_df], ignore_index=True)
            combined_df = combined_df.drop_duplicates(subset=['id'])
        except Exception as e:
            if pbar:
                pbar.write(f"파케이 파일 읽기 오류: {e}")
            combined_df = new_df
    else:
        combined_df = new_df

    combined_df.to_parquet(SAVE_PATH, index=False)
    
    if pbar:
        pbar.set_postfix({"누적저장": f"{len(combined_df)}건"})
        
    buffer.clear()

def worker(pbar):
    driver = create_driver()
    try:
        while True:
            post_id = task_queue.get()
            if post_id is None:
                task_queue.task_done()
                break
            
            try:
                target_url = f"https://safebooru.org/index.php?page=post&s=view&id={post_id}"
                driver.get(target_url)
                # 요청 간격 제거 (사용자 요청 사항)
                
                try:
                    img_element = driver.find_element(By.ID, "image")
                    tags = img_element.get_attribute("alt").strip()
                    
                    original_link_element = driver.find_element(By.XPATH, "//a[contains(text(), 'Original image')]")
                    file_url = original_link_element.get_attribute("href")
                    
                    size_element = driver.find_element(By.XPATH, "//div[@id='stats']//li[contains(text(), 'Size:')]")
                    size_text = size_element.text.replace("Size:", "").strip()
                    width, height = size_text.split("x")
                    
                    result_data = {
                        "id": post_id,
                        "tags": tags,
                        "file_url": file_url,
                        "width": int(width),
                        "height": int(height)
                    }
                    
                    with save_lock:
                        buffer.append(result_data)
                        if len(buffer) >= SAVE_INTERVAL:
                            save_buffer(pbar)
                            
                except NoSuchElementException:
                    pass
                    
            except Exception as e:
                pbar.write(f"[에러] ID {post_id} 파싱 실패: {e}")
            finally:
                pbar.update(1)
                task_queue.task_done()
    finally:
        driver.quit()

if __name__ == "__main__":
    print(f"=== 크롤링 시작 ===")
    print(f"저장 경로: {SAVE_PATH}")
    
    total_tasks = START_ID - END_ID + 1
    pbar = tqdm(total=total_tasks, desc="수집 진행률", ncols=100)
    
    threads = []
    for _ in range(NUM_THREADS):
        t = threading.Thread(target=worker, args=(pbar,))
        t.daemon = True
        t.start()
        threads.append(t)
        
    try:
        for pid in range(START_ID, END_ID - 1, -1):
            task_queue.put(pid) 
        
        task_queue.join() 
    except KeyboardInterrupt:
        print("\n사용자에 의해 중단되었습니다. 남은 데이터를 저장합니다.")
    
    for _ in range(NUM_THREADS):
        task_queue.put(None)
    for t in threads:
        t.join()
        
    with save_lock:
        if buffer:
            save_buffer(pbar)

    pbar.close()
    print("\n=== 모든 크롤링 작업이 완료되었습니다. ===")

=== 크롤링 시작 ===
저장 경로: ../data/metadata_html.parquet


수집 진행률:   1%|                  | 37662/6654626 [1:18:01<332:10:04,  5.53it/s, 누적저장=48884건]it/s]